# Daily Merge

Merge raw session CSV files into one `merged_<animal>.csv` file per animal, then optionally merge all animals into `merged_all_subjects.csv` for the full cohort of the selected line.

## 1. Setup

Run this cell first. It makes imports work whether the notebook is launched from the repo root or from inside the `notebooks/` folder.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

## 2. Choose Dataset

Set `RAT = None` to process every animal in the cohort, or set it to one animal ID such as `"ASD0026"`.

In [ ]:
LINE = "SHANK3"
COHORT = "cohort1"
RAT = None  # e.g. "ASD0026", or None for all animals

MODE = "both"  # "session", "animals", or "both"
MODEL_FILE = None

## 3. Optional Session Removals

Edit `SESSION_EDITS` before merging if a daily CSV should be removed entirely, or if only the bad tail/range of a session should be removed or marked repeated. File names must match the raw CSV names inside each animal folder.


In [ ]:
# Optional per-animal cleanup rules applied while creating merged_<animal>.csv.
# These affect the merged output only; they do not edit the raw daily CSV files.
#
# Actions:
# - drop_entire_session: skip that raw CSV completely
# - drop_from_trial: remove trials with trial >= start_trial
# - drop_trial_range: remove trials from start_trial through end_trial, inclusive
# - mark_repeated_from: keep rows but set repeated_trial = True from start_trial onward

SESSION_EDITS = {
    # Previous provisions from DailyMerge.py.
    "ASD0013": [
        {"file": "out_ASD0013_251014.csv", "action": "mark_repeated_from", "start_trial": 6690},
    ],

    # Previous ASD0018 provisions from DailyMerge.py.
    # Change action to "drop_from_trial" if you want these rows removed instead.
    "ASD0018": [
        {"file": "ASD0018_out_251014.csv", "action": "mark_repeated_from", "start_trial": 7370},
        {"file": "ASD0018_out_251015.csv", "action": "mark_repeated_from", "start_trial": 8000},
        {"file": "out_ASD0018_251028.csv", "action": "mark_repeated_from", "start_trial": 10900},
        {"file": "out_ASD0018_251127.csv", "action": "mark_repeated_from", "start_trial": 22250},
    ],

    # Examples for ASD0019. Uncomment/edit the raw filenames and thresholds as needed.
    # "ASD0019": [
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_entire_session"},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_from_trial", "start_trial": 5000},
    #     {"file": "out_ASD0019_YYMMDD.csv", "action": "drop_trial_range", "start_trial": 1000, "end_trial": 1500},
    # ],
}

SESSION_EDITS


## 4. Optional Bad RT Values

Use `RT_VALUE_EDITS` when task outcomes and abort labels are valid, but the recorded numeric `timed_rt` values should be ignored in RT analyses. These edits keep the trials and only set `timed_rt` to missing in the merged outputs.


In [ ]:
# Numeric RT values to ignore while keeping trials for accuracy/choice/abort analyses.
# This only blanks timed_rt and adds rt_value_valid / rt_value_note columns.
# It does not change success, abort_type, choices, trial counts, or repeated_trial.

RT_VALUE_EDITS = [
    {
        "setup": 2,
        "start_date": "2026-06-13",
        "end_date": "2026-06-18",  # update if the setup-2 issue continues
        "date_col": "source_date",
        "setup_col": "box",
        "rt_col": "timed_rt",
        "reason": "setup 2 RT value recording issue",
    },
]

RT_VALUE_EDITS


## 5. Preview Animals

Check which animals will be processed before writing merged files.

In [ ]:
import importlib
import DailyMerge
importlib.reload(DailyMerge)
from DailyMerge import get_animals_for_cohort, get_base_dir

base_dir = get_base_dir(LINE, COHORT)
animals = get_animals_for_cohort(LINE, COHORT, rat=RAT)

print(f"Base directory: {base_dir}")
print(f"Animals ({len(animals)}): {animals}")

## 6. Merge Daily Files Per Animal

This creates or updates `merged_<animal>.csv` files in the cohort data folder.

In [ ]:
import importlib
import DailyMerge
importlib.reload(DailyMerge)
from DailyMerge import merge_session_files

if MODE in ("session", "both"):
    merge_session_files(
        line=LINE,
        cohort=COHORT,
        rat=RAT,
        session_edits=SESSION_EDITS,
        rt_value_edits=RT_VALUE_EDITS,
    )
else:
    print("Skipping per-animal session merge.")

## 7. Merge Animals Into Cohort File

This creates or updates `merged_all_subjects.csv` in the cohort data folder.

In [ ]:
import importlib
import DailyMerge
importlib.reload(DailyMerge)
from DailyMerge import merge_subject_files_with_model

if MODE in ("animals", "both"):
    merged_df = merge_subject_files_with_model(
        line=LINE,
        cohort=COHORT,
        model_file=MODEL_FILE,
    )
else:
    merged_df = None
    print("Skipping cohort-level animal merge.")

if merged_df is not None:
    display(merged_df.head())
    print(merged_df.shape)